# Feature Analysis

**Forensic question:** How do writing features evolve over time, and when do they deviate from baseline?

**Input artifacts:**
- `data/features/*.parquet`

**Output artifacts:**
- (figures)

**Run metadata:** (auto-populated by first code cell)


In [ ]:
# | echo: false
import sys
from datetime import UTC, datetime

from IPython.display import Markdown, display

from forensics.config import settings
from forensics.utils.provenance import compute_config_hash, compute_corpus_hash

config_hash = compute_config_hash(settings)
corpus_hash = compute_corpus_hash(settings.db_path)
run_timestamp = datetime.now(UTC).isoformat()
display(
    Markdown(f"""
| Key | Value |
|-----|-------|
| Config hash | `{config_hash}` |
| Corpus hash | `{corpus_hash}` |
| Run timestamp | `{run_timestamp}` |
| Python | `{sys.version}` |
""")
)

In [ ]:
import polars as pl
from IPython.display import display

from forensics.config import get_project_root, settings
from forensics.storage.parquet import read_features

root = get_project_root()
slug = settings.authors[0].slug
p = root / "data" / "features" / f"{slug}.parquet"
if p.is_file():
    df = read_features(p)
    num = [c for c in df.columns if df[c].dtype in (pl.Float32, pl.Float64, pl.Int32, pl.Int64)]
    num = [c for c in num if c not in ("timestamp",)]
    sample = num[:12]
    cm = df.select(sample).corr()
    display(cm)
else:
    print("Missing features parquet for", slug)

**Summary finding:** Feature correlations and time-series panels link lexical shifts to timing.
